<a href="https://colab.research.google.com/github/shikhar-iimc/Prajna-previous_Versions/blob/main/Prajna_Demo_Main_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# PRAJNA — Strategic Intelligence Engine
# Cell 1: Install & Import
# ============================================
!pip install requests pandas neo4j groq spacy pyvis streamlit trafilatura pyngrok -q
!pip install newspaper3k lxml_html_clean -q
!python -m spacy download en_core_web_sm -q

import requests, json, spacy, trafilatura, concurrent.futures, time
import os
from trafilatura.settings import use_config
from neo4j import GraphDatabase
from google.colab import userdata
from datetime import datetime, timezone
from groq import Groq
from collections import defaultdict

# ── Credentials ──
NEWSAPI_KEY  = userdata.get("NEWSAPI_KEY")
NEO4J_URI    = userdata.get("NEO4J_URI")
NEO4J_USER   = userdata.get("NEO4J_USERNAME")
NEO4J_PASS   = userdata.get("NEO4J_PASSWORD")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

# ── Models + Connections ──
nlp    = spacy.load("en_core_web_sm")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
client = Groq(api_key=GROQ_API_KEY)

def get_week_key():
    now = datetime.now(timezone.utc)
    return f"{now.year}-W{now.strftime('%W')}"

print("✅ All packages installed and imported")
print(f"📅 Current week: {get_week_key()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All packages installed and imported
📅 Current week: 2026-W09


In [ ]:
# ============================================
# PRAJNA — Cell 2: Multi-Source Data Ingestion
# Sources: NewsAPI + GDELT + Guardian API
# Never stops — automatic fallback between sources
# ============================================

import requests, json, spacy, trafilatura, concurrent.futures, time
from trafilatura.settings import use_config
from collections import defaultdict
from datetime import datetime, timezone

nlp = spacy.load("en_core_web_sm")

# ── Credentials ──
NEWSAPI_KEY   = userdata.get("NEWSAPI_KEY")
GUARDIAN_KEY  = userdata.get("GUARDIAN_KEY")   # optional — add to Colab secrets if you have it

# ── Trafilatura fast config ──
traf_config = use_config()
traf_config.set("DEFAULT", "DOWNLOAD_TIMEOUT", "5")
traf_config.set("DEFAULT", "MAX_REDIRECTS", "2")

# ── Entity normalization ──
ENTITY_ALIASES = {
    "Indian":"India","Indians":"India","Bharatiya":"India",
    "US":"United States","U.S.":"United States","U.S.A.":"United States","America":"United States",
    "Pakistani":"Pakistan","Pakistanis":"Pakistan",
    "Iranian":"Iran","Iranians":"Iran",
    "Israeli":"Israel","Israelis":"Israel",
    "Chinese":"China","Taiwanese":"Taiwan",
    "Russian":"Russia","Russians":"Russia",
    "Afghan":"Afghanistan","Afghans":"Afghanistan",
    "Saudi":"Saudi Arabia",
    "UAE":"United Arab Emirates","Emirati":"United Arab Emirates",
    "British":"United Kingdom","UK":"United Kingdom",
    "Donald Trump":"Trump",
    "Washington":"United States",
    "Beijing":"China",
    "Moscow":"Russia",
    "Tehran":"Iran",
    "Islamabad":"Pakistan",
    "New Delhi":"India",
    "Kyiv":"Ukraine","Kiev":"Ukraine",
}

def normalize_entity(text):
    return ENTITY_ALIASES.get(text.strip(), text.strip())

VALID_LABELS = {"GPE", "ORG", "PERSON", "NORP", "LOC", "EVENT"}

# ── Topics ──
DOMAIN_TOPICS = {
    "GEOPOLITICS": [
        "India geopolitics 2026",
        "India China relations",
        "India Pakistan border",
        "India Iran oil energy",
        "India Russia defense",
        "India Middle East policy",
        "Taliban Afghanistan conflict",
        "Israel Iran war",
        "India USA relations 2026",
        "India Bangladesh relations",
        "BRICS India 2026",
        "Quad alliance Indo-Pacific",
        "India Nepal border",
    ],
    "DOMESTIC": [
        "India elections 2026",
        "India parliament budget 2026",
        "India BJP Congress politics",
        "India supreme court judgment",
    ],
    "ECONOMICS": [
        "India economy GDP 2026",
        "India stock market Sensex",
        "India trade exports",
        "India inflation RBI policy",
        "India FDI foreign investment 2026",
    ],
    "TECHNOLOGY": [
        "India semiconductor chip manufacturing",
        "India AI policy",
        "India space ISRO 2026",
        "India defence procurement",
        "India cybersecurity threat",
    ],
    "SOCIETY": [
        "India infrastructure smart city",
        "India climate environment",
        "India health policy",
        "India startup ecosystem 2026",
        "India education skill development",
    ],
}

# ════════════════════════════════════════════
# SOURCE 1 — NewsAPI
# ════════════════════════════════════════════
def fetch_newsapi(topic, domain, page_size=10):
    try:
        r = requests.get("https://newsapi.org/v2/everything", params={
            "q":        topic,
            "language": "en",
            "sortBy":   "publishedAt",
            "pageSize": page_size,
            "apiKey":   NEWSAPI_KEY
        }, timeout=10)
        if r.status_code == 200:
            data = r.json()
            if data.get("status") == "ok":
                results = []
                for a in data.get("articles", []):
                    results.append({
                        "title":       a.get("title", ""),
                        "url":         a.get("url", ""),
                        "publishedAt": a.get("publishedAt", ""),
                        "source":      {"name": a.get("source", {}).get("name", "NewsAPI")},
                        "description": a.get("description", ""),
                        "domain":      domain
                    })
                return results, "newsapi"
            elif "rateLimited" in str(data) or data.get("code") == "rateLimited":
                return None, "rate_limited"
        elif r.status_code == 429:
            return None, "rate_limited"
    except Exception as e:
        pass
    return None, "error"

# ════════════════════════════════════════════
# SOURCE 2 — GDELT (no key, no limits)
# ════════════════════════════════════════════
def fetch_gdelt(topic, domain, max_records=15):
    try:
        r = requests.get(
            "https://api.gdeltproject.org/api/v2/doc/doc",
            params={
                "query":      topic,
                "mode":       "artlist",
                "maxrecords": max_records,
                "format":     "json",
                "timespan":   "1week",
                "sort":       "datedesc"
            }, timeout=15
        )
        if r.status_code == 200:
            data = r.json()
            results = []
            for a in data.get("articles", []):
                results.append({
                    "title":       a.get("title", ""),
                    "url":         a.get("url", ""),
                    "publishedAt": a.get("seendate", ""),
                    "source":      {"name": a.get("domain", "GDELT")},
                    "description": "",
                    "domain":      domain
                })
            return results, "gdelt"
    except Exception as e:
        pass
    return None, "error"

# ════════════════════════════════════════════
# SOURCE 3 — The Guardian (free, 5000/day)
# ════════════════════════════════════════════
def fetch_guardian(topic, domain, page_size=10):
    try:
        key = GUARDIAN_KEY if GUARDIAN_KEY else "test"  # test key = limited but works
        r = requests.get(
            "https://content.guardianapis.com/search",
            params={
                "q":          topic,
                "lang":       "en",
                "page-size":  page_size,
                "order-by":   "newest",
                "show-fields":"headline,trailText,shortUrl",
                "api-key":    key
            }, timeout=10
        )
        if r.status_code == 200:
            results = []
            for a in r.json().get("response", {}).get("results", []):
                fields = a.get("fields", {})
                results.append({
                    "title":       fields.get("headline", a.get("webTitle", "")),
                    "url":         a.get("webUrl", ""),
                    "publishedAt": a.get("webPublicationDate", ""),
                    "source":      {"name": "The Guardian"},
                    "description": fields.get("trailText", ""),
                    "domain":      domain
                })
            return results, "guardian"
    except Exception as e:
        pass
    return None, "error"

# ════════════════════════════════════════════
# SMART FETCHER — tries all sources, never stops
# ════════════════════════════════════════════
def fetch_with_fallback(topic, domain):
    """
    Try NewsAPI first.
    If rate limited → fall back to GDELT + Guardian.
    Always returns articles from at least one source.
    """
    # Try NewsAPI
    results, status = fetch_newsapi(topic, domain)
    if results:
        return results, "newsapi"

    if status == "rate_limited":
        # NewsAPI exhausted — use free sources
        gdelt_results, _    = fetch_gdelt(topic, domain)
        guardian_results, _ = fetch_guardian(topic, domain)

        combined = []
        if gdelt_results:    combined.extend(gdelt_results)
        if guardian_results: combined.extend(guardian_results)
        if combined:
            return combined, "fallback (gdelt+guardian)"

    # Last resort — GDELT alone
    gdelt_results, _ = fetch_gdelt(topic, domain)
    if gdelt_results:
        return gdelt_results, "gdelt"

    return [], "no_results"

# ════════════════════════════════════════════
# SCRAPER
# ════════════════════════════════════════════
def scrape_full_text(url):
    try:
        downloaded = trafilatura.fetch_url(url, config=traf_config)
        if downloaded:
            return trafilatura.extract(downloaded, config=traf_config)
    except:
        pass
    return None

# ════════════════════════════════════════════
# MAIN INGESTION
# ════════════════════════════════════════════

# ── Clear only current week ──
week_key = get_week_key()
with driver.session() as session:
    deleted = session.run(
        "MATCH (a:Article {week:$week}) DETACH DELETE a RETURN count(a) as c",
        {"week": week_key}
    ).single()["c"]
print(f"🗑️  Cleared {deleted} articles for {week_key} — historical data preserved\n")

# ── Fetch from all sources ──
articles   = []
seen_urls  = set()
source_log = defaultdict(int)

total_topics = sum(len(t) for t in DOMAIN_TOPICS.values())
print(f"🔍 Fetching {total_topics} topics across {len(DOMAIN_TOPICS)} domains...\n")

for domain, topics in DOMAIN_TOPICS.items():
    domain_count = 0
    for topic in topics:
        results, source = fetch_with_fallback(topic, domain)
        new_count = 0
        for a in results:
            url = a.get("url", "")
            if url and url not in seen_urls:
                seen_urls.add(url)
                articles.append(a)
                new_count += 1
        if new_count:
            source_log[source] += new_count
            domain_count += new_count
        time.sleep(0.15)
    print(f"  ✅ {domain}: {domain_count} articles")

print(f"\n📊 SOURCE BREAKDOWN:")
for source, count in sorted(source_log.items(), key=lambda x: -x[1]):
    print(f"   {source:<30} {count} articles")
print(f"\n✅ Total unique articles: {len(articles)}")

# ── Parallel full-text scrape ──
print(f"\n🌐 Scraping full text (parallel, 5s timeout)...")
full_texts = {}
urls = [a.get("url", "") for a in articles if a.get("url")]

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    future_map = {executor.submit(scrape_full_text, url): url for url in urls}
    completed  = 0
    for future in concurrent.futures.as_completed(future_map):
        url = future_map[future]
        try:
            text = future.result(timeout=8)
            if text:
                full_texts[url] = text
        except:
            pass
        completed += 1
        if completed % 50 == 0:
            print(f"  📄 {completed}/{len(urls)} scraped ({len(full_texts)} successful)")

print(f"  ✅ Full text: {len(full_texts)}/{len(urls)} articles scraped")

# ── Extract entities ──
print(f"\n🧠 Extracting entities...")
entities_per_article = []
for article in articles:
    url  = article.get("url", "")
    text = full_texts.get(url) or \
           f"{article.get('title', '')} {article.get('description', '')}"
    doc  = nlp(text[:5000])
    ents = []
    seen = set()
    for ent in doc.ents:
        if ent.label_ not in VALID_LABELS: continue
        n = normalize_entity(ent.text.strip())
        if len(n) < 2 or len(n) > 40 or n in seen: continue
        seen.add(n)
        ents.append((n, ent.label_))
    entities_per_article.append(ents)

total_mentions = sum(len(e) for e in entities_per_article)
print(f"✅ {total_mentions} entity mentions extracted from {len(articles)} articles")
print(f"\n🎉 Ingestion complete — ready for Cell 3")

🗑️  Cleared 0 articles for 2026-W09 — historical data preserved

🔍 Fetching 32 topics across 5 domains...

  ✅ GEOPOLITICS: 56 articles
  ✅ DOMESTIC: 7 articles
  ✅ ECONOMICS: 11 articles
  ✅ TECHNOLOGY: 15 articles
  ✅ SOCIETY: 19 articles

📊 SOURCE BREAKDOWN:
   fallback (gdelt+guardian)      108 articles

✅ Total unique articles: 108

🌐 Scraping full text (parallel, 5s timeout)...


  📄 50/108 scraped (50 successful)


  📄 100/108 scraped (100 successful)
  ✅ Full text: 108/108 articles scraped

🧠 Extracting entities...
✅ 2846 entity mentions extracted from 108 articles

🎉 Ingestion complete — ready for Cell 3


In [ ]:
# ============================================
# PRAJNA — Cell 3: Populate Neo4j Graph
# Fixed: proper JSON merging, no string concat
# Safe to re-run — current week cleanly overwritten
# ============================================

from collections import defaultdict

week_key = get_week_key()
print(f"📅 Populating graph for {week_key}")

# ════════════════════════════════════════════
# STEP 1 — Reset this week's data
# ════════════════════════════════════════════
print("\n♻️  Resetting current week data...")

with driver.session() as session:
    result = session.run("""
        MATCH (a:Entity)-[r:CO_OCCURS_WITH]-(b:Entity)
        WHERE r.weekly_counts_json IS NOT NULL
        RETURN id(r) as rid, r.weekly_counts_json as wcj, r.count as total
    """).data()

    updates = []
    for row in result:
        try:
            import re
            wcj_str = row["wcj"] or "{}"
            pairs   = re.findall(r'"(2026-W\d+)"\s*:\s*(\d+)', wcj_str)
            wc      = {k: int(v) for k, v in pairs}
        except:
            wc = {}
        if week_key in wc:
            new_wc = {k: v for k, v in wc.items() if k != week_key}
            updates.append({
                "rid":      row["rid"],
                "subtract": wc[week_key],
                "new_wcj":  json.dumps(new_wc)
            })

    if updates:
        for i in range(0, len(updates), 500):
            session.run("""
                UNWIND $batch AS row
                MATCH ()-[r]-() WHERE id(r) = row.rid
                SET r.count = CASE
                        WHEN r.count - row.subtract < 0 THEN 0
                        ELSE r.count - row.subtract
                    END,
                    r.weekly_counts_json = row.new_wcj
            """, {"batch": updates[i:i+500]})
        print(f"   ✅ Reset {len(updates)} relationships for {week_key}")
    else:
        print(f"   ✅ No existing {week_key} data — clean slate")

# ════════════════════════════════════════════
# STEP 2 — Reconnect Neo4j
# ════════════════════════════════════════════
driver.close()
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print("\n✅ Neo4j reconnected")

# ════════════════════════════════════════════
# STEP 3 — Collect entity pairs
# ════════════════════════════════════════════
pair_counts  = defaultdict(int)
pair_domains = defaultdict(lambda: defaultdict(int))

for article, entities in zip(articles, entities_per_article):
    unique = list(set([e[0] for e in entities]))
    domain = article.get("domain", "GENERAL")
    for i in range(len(unique)):
        for j in range(i + 1, len(unique)):
            key = tuple(sorted([unique[i], unique[j]]))
            pair_counts[key]          += 1
            pair_domains[key][domain] += 1

print(f"\n📊 {len(pair_counts)} unique entity pairs found")

# ════════════════════════════════════════════
# STEP 4 — Fetch existing weekly_counts_json
# for all affected pairs (Python-side merge)
# ════════════════════════════════════════════
print("🔍 Fetching existing weekly data for merge...")

existing_wcj = {}
all_names    = list(set(
    [p[0] for p in pair_counts] +
    [p[1] for p in pair_counts]
))

# Fetch in chunks of 500 names
import re
for chunk_start in range(0, len(all_names), 500):
    chunk = all_names[chunk_start:chunk_start+500]
    with driver.session() as session:
        rows = session.run("""
            MATCH (a:Entity)-[r:CO_OCCURS_WITH]-(b:Entity)
            WHERE a.name IN $names AND b.name IN $names
            RETURN a.name AS e1, b.name AS e2,
                   r.weekly_counts_json AS wcj
        """, {"names": chunk}).data()
        for row in rows:
            key = tuple(sorted([row["e1"], row["e2"]]))
            try:
                wcj_str = row["wcj"] or "{}"
                pairs   = re.findall(r'"(2026-W\d+)"\s*:\s*(\d+)', wcj_str)
                existing_wcj[key] = {k: int(v) for k, v in pairs}
            except:
                existing_wcj[key] = {}

print(f"   ✅ Found existing data for {len(existing_wcj)} pairs")

# ════════════════════════════════════════════
# STEP 5 — Write everything to Neo4j
# ════════════════════════════════════════════
print(f"\n📝 Writing to Neo4j...")

with driver.session() as session:

    # ── Articles ──
    session.run("""
        UNWIND $batch AS row
        MERGE (a:Article {url: row.url})
        SET a.title     = row.title,
            a.source    = row.source,
            a.published = row.published,
            a.week      = row.week,
            a.domain    = row.domain
    """, {"batch": [{
        "url":       a.get("url", "unknown"),
        "title":     a.get("title", ""),
        "source":    a.get("source", {}).get("name", "Unknown"),
        "published": a.get("publishedAt", ""),
        "week":      week_key,
        "domain":    a.get("domain", "GENERAL")
    } for a in articles]})
    print(f"   ✅ {len(articles)} articles written")

    # ── Entities + MENTIONS ──
    mentions = []
    for article, entities in zip(articles, entities_per_article):
        for name, label in entities:
            mentions.append({
                "url":  article.get("url", "unknown"),
                "name": name,
                "type": label
            })
    for i in range(0, len(mentions), 500):
        session.run("""
            UNWIND $batch AS row
            MERGE (e:Entity {name: row.name})
            SET e.type = row.type
            WITH e, row
            MATCH (a:Article {url: row.url})
            MERGE (a)-[:MENTIONS]->(e)
        """, {"batch": mentions[i:i+500]})
    print(f"   ✅ {len(mentions)} entity mentions written")

    # ── CO_OCCURS_WITH pairs ──
    # Merge weekly counts in Python — always valid JSON
    pairs_list = list(pair_counts.items())
    for batch_start in range(0, len(pairs_list), 200):
        batch = pairs_list[batch_start:batch_start+200]
        session.run("""
            UNWIND $batch AS row
            MATCH (a:Entity {name: row.e1})
            MATCH (b:Entity {name: row.e2})
            MERGE (a)-[r:CO_OCCURS_WITH]-(b)
            ON CREATE SET
                r.count              = row.cnt,
                r.first_seen         = row.week,
                r.weekly_counts_json = row.wcj,
                r.domains_json       = row.dj
            ON MATCH SET
                r.count              = r.count + row.cnt,
                r.weekly_counts_json = row.wcj,
                r.domains_json       = row.dj
        """, {"batch": [{
            "e1":   pair[0],
            "e2":   pair[1],
            "cnt":  count,
            "week": week_key,
            "wcj":  json.dumps({
                        **existing_wcj.get(pair, {}),
                        week_key: count
                    }),
            "dj":   json.dumps(dict(pair_domains[pair]))
        } for pair, count in batch]})

        if batch_start % 2000 == 0 and batch_start > 0:
            print(f"   ✅ Pairs: {batch_start}/{len(pairs_list)}")

print(f"   ✅ {len(pairs_list)} pairs written")

# ════════════════════════════════════════════
# STEP 6 — Verify
# ════════════════════════════════════════════
print(f"\n📊 GRAPH STATUS:")
with driver.session() as session:
    a = session.run("MATCH (a:Article) RETURN count(a) as c").single()["c"]
    e = session.run("MATCH (e:Entity)  RETURN count(e) as c").single()["c"]
    r = session.run("MATCH ()-[r:CO_OCCURS_WITH]->() RETURN count(r) as c").single()["c"]
    print(f"   Articles:      {a}")
    print(f"   Entities:      {e}")
    print(f"   Relationships: {r}")

    print(f"\n📅 WEEKS IN GRAPH:")
    for row in session.run("""
        MATCH (a:Article)
        RETURN a.week as week, count(a) as c
        ORDER BY week
    """):
        print(f"   {row['week']}: {row['c']} articles")

print(f"\n🎉 Done — {week_key} populated cleanly")

📅 Populating graph for 2026-W09

♻️  Resetting current week data...


   ✅ Reset 76028 relationships for 2026-W09

✅ Neo4j reconnected

📊 38224 unique entity pairs found
🔍 Fetching existing weekly data for merge...
   ✅ Found existing data for 13431 pairs

📝 Writing to Neo4j...
   ✅ 108 articles written
   ✅ 2846 entity mentions written
   ✅ Pairs: 2000/38224
   ✅ Pairs: 4000/38224
   ✅ Pairs: 6000/38224
   ✅ Pairs: 8000/38224
   ✅ Pairs: 10000/38224
   ✅ Pairs: 12000/38224
   ✅ Pairs: 14000/38224
   ✅ Pairs: 16000/38224
   ✅ Pairs: 18000/38224
   ✅ Pairs: 20000/38224
   ✅ Pairs: 22000/38224
   ✅ Pairs: 24000/38224
   ✅ Pairs: 26000/38224
   ✅ Pairs: 28000/38224
   ✅ Pairs: 30000/38224
   ✅ Pairs: 32000/38224
   ✅ Pairs: 34000/38224
   ✅ Pairs: 36000/38224
   ✅ Pairs: 38000/38224
   ✅ 38224 pairs written

📊 GRAPH STATUS:
   Articles:      690
   Entities:      9274
   Relationships: 268790

📅 WEEKS IN GRAPH:
   2026-W05: 164 articles
   2026-W06: 174 articles
   2026-W07: 143 articles
   2026-W08: 101 articles
   2026-W09: 108 articles

🎉 Done — 2026-W09

In [ ]:
# ============================================
# PRAJNA — Cell 4: Cleanup + Stats
# Run once after Cell 3
# ============================================

with driver.session() as session:
    # Fix known misclassifications
    session.run("MATCH (e:Entity {name:'China'}) SET e.type='GPE'")
    session.run("MATCH (e:Entity {name:'India'}) SET e.type='GPE'")
    session.run("MATCH (e:Entity {name:'Trump'}) SET e.type='PERSON'")
    session.run("MATCH (e:Entity {name:'AI', type:'GPE'}) DETACH DELETE e")

    # Merge duplicates
    for keep, dup in [
        ("United States", "Washington"),
        ("United States", "America"),
        ("Iran", "Tehran"),
        ("China", "Beijing"),
        ("Russia", "Moscow"),
        ("Pakistan", "Islamabad"),
        ("India", "New Delhi"),
    ]:
        session.run("""
            MATCH (keep:Entity {name:$keep})
            MATCH (dup:Entity {name:$dup})
            WITH keep, dup
            MATCH (dup)-[r:CO_OCCURS_WITH]-(other:Entity)
            WHERE other <> keep
            MERGE (keep)-[r2:CO_OCCURS_WITH]-(other)
            ON CREATE SET r2.count=r.count,
                          r2.weekly_counts_json=r.weekly_counts_json,
                          r2.domains_json=r.domains_json
            ON MATCH SET  r2.count=r2.count+r.count
            WITH dup DETACH DELETE dup
        """, {"keep": keep, "dup": dup})

    # Fix Strait of Hormuz
    session.run("""
        MATCH (e:Entity) WHERE e.name STARTS WITH 'the '
        SET e.name = substring(e.name, 4)
    """)

print("✅ Cleanup done\n")

# ── Stats ──
with driver.session() as session:
    articles_c = session.run("MATCH (a:Article) RETURN count(a) as c").single()["c"]
    entities_c = session.run("MATCH (e:Entity) RETURN count(e) as c").single()["c"]
    rels_c     = session.run("MATCH ()-[r:CO_OCCURS_WITH]->() RETURN count(r) as c").single()["c"]

    print(f"📊 GRAPH STATUS:")
    print(f"   Articles:      {articles_c}")
    print(f"   Entities:      {entities_c}")
    print(f"   Relationships: {rels_c}")

    print(f"\n🌍 TOP 15 ENTITIES:")
    result = session.run("""
        MATCH (e:Entity)
        WITH e, COUNT{(e)--()} as conn
        ORDER BY conn DESC LIMIT 15
        RETURN e.name as name, e.type as type, conn
    """)
    for r in result:
        print(f"   {r['name']:<28} {r['type']:<10} {r['conn']}")

    print(f"\n📅 WEEKS IN GRAPH:")
    result = session.run("""
        MATCH (a:Article)
        RETURN a.week as week, count(a) as articles
        ORDER BY week
    """)
    for r in result:
        print(f"   {r['week']}: {r['articles']} articles")

✅ Cleanup done

📊 GRAPH STATUS:
   Articles:      108
   Entities:      1817
   Relationships: 38014

🌍 TOP 15 ENTITIES:
   United States                GPE        1060
   Iran                         GPE        838
   United Kingdom               GPE        783
   Trump                        PERSON     731
   Middle East                  LOC        595
   Israel                       NORP       582
   Guardian                     ORG        442
   London                       GPE        382
   Australia                    GPE        375
   China                        GPE        370
   Donald Trump’s               PERSON     361
   American                     NORP       349
   Russia                       GPE        340
   Australian                   NORP       324
   Gulf                         LOC        311

📅 WEEKS IN GRAPH:
   2026-W09: 108 articles


In [54]:
# ============================================
# PRAJNA — Cell 5: Historical Backfill
# Multi-source: NewsAPI + GDELT + Guardian
# Never stops — automatic fallback
# Run once after Cell 3
# ============================================

from datetime import timedelta

def get_week_key_for_date(date):
    return f"{date.year}-W{date.strftime('%W')}"

# ── Generate last 4 weeks ──
today   = datetime.now(timezone.utc)
windows = []
start   = today - timedelta(days=28)
while start < today - timedelta(days=1):
    end = start + timedelta(days=6)
    windows.append((
        start.strftime("%Y-%m-%d"),
        end.strftime("%Y-%m-%d"),
        get_week_key_for_date(start)
    ))
    start += timedelta(days=7)

# ── Check which weeks already exist ──
with driver.session() as session:
    existing = {r["week"] for r in
                session.run("MATCH (a:Article) RETURN DISTINCT a.week as week").data()
                if r["week"]}

print(f"📅 Weeks to backfill: {[w for _,_,w in windows]}")
print(f"⏭️  Already have:     {existing}")
to_fetch = [(f, t, w) for f, t, w in windows if w not in existing]
print(f"📥 Will fetch:        {[w for _,_,w in to_fetch]}\n")

if not to_fetch:
    print("✅ All weeks already in graph — nothing to backfill")
else:
    BACKFILL_TOPICS = [
        ("India geopolitics 2026",        "GEOPOLITICS"),
        ("India China relations",          "GEOPOLITICS"),
        ("India Pakistan border",          "GEOPOLITICS"),
        ("India Iran oil energy",          "GEOPOLITICS"),
        ("India Russia defense",           "GEOPOLITICS"),
        ("Israel Iran war",                "GEOPOLITICS"),
        ("Taliban Afghanistan conflict",   "GEOPOLITICS"),
        ("India USA relations",            "GEOPOLITICS"),
        ("India economy GDP",              "ECONOMICS"),
        ("India semiconductor chip",       "TECHNOLOGY"),
        ("India defence procurement",      "TECHNOLOGY"),
        ("India elections politics",       "DOMESTIC"),
    ]

    # ════════════════════════════════════════
    # BACKFILL SOURCE 1 — NewsAPI (date range)
    # ════════════════════════════════════════
    def backfill_newsapi(topic, domain, from_date, to_date):
        try:
            r = requests.get("https://newsapi.org/v2/everything", params={
                "q":        topic,
                "language": "en",
                "from":     from_date,
                "to":       to_date,
                "sortBy":   "publishedAt",
                "pageSize": 20,
                "apiKey":   NEWSAPI_KEY
            }, timeout=10)
            if r.status_code == 200:
                data = r.json()
                if data.get("status") == "ok":
                    return [{
                        "title":       a.get("title", ""),
                        "url":         a.get("url", ""),
                        "publishedAt": a.get("publishedAt", ""),
                        "source":      {"name": a.get("source", {}).get("name", "NewsAPI")},
                        "description": a.get("description", ""),
                        "domain":      domain
                    } for a in data.get("articles", [])], "newsapi"
                elif data.get("code") in ("rateLimited", "maximumResultsReached"):
                    return None, "rate_limited"
            elif r.status_code == 429:
                return None, "rate_limited"
        except:
            pass
        return None, "error"

    # ════════════════════════════════════════
    # BACKFILL SOURCE 2 — GDELT (date range)
    # ════════════════════════════════════════
    def backfill_gdelt(topic, domain, from_date, to_date):
        try:
            # GDELT date format: YYYYMMDDHHMMSS
            start_dt = from_date.replace("-", "") + "000000"
            end_dt   = to_date.replace("-", "") + "235959"
            r = requests.get(
                "https://api.gdeltproject.org/api/v2/doc/doc",
                params={
                    "query":         f"{topic} sourcelang:english",
                    "mode":          "artlist",
                    "maxrecords":    50,
                    "format":        "json",
                    "startdatetime": start_dt,
                    "enddatetime":   end_dt,
                    "sort":          "datedesc"
                }, timeout=15
            )
            if r.status_code == 200:
                data = r.json()
                return [{
                    "title":       a.get("title", ""),
                    "url":         a.get("url", ""),
                    "publishedAt": a.get("seendate", ""),
                    "source":      {"name": a.get("domain", "GDELT")},
                    "description": "",
                    "domain":      domain
                } for a in data.get("articles", [])], "gdelt"
        except:
            pass
        return None, "error"

    # ════════════════════════════════════════
    # BACKFILL SOURCE 3 — Guardian (date range)
    # ════════════════════════════════════════
    def backfill_guardian(topic, domain, from_date, to_date):
        try:
            key = GUARDIAN_KEY if GUARDIAN_KEY else "test"
            r = requests.get(
                "https://content.guardianapis.com/search",
                params={
                    "q":            topic,
                    "lang":         "en",
                    "from-date":    from_date,
                    "to-date":      to_date,
                    "page-size":    20,
                    "order-by":     "newest",
                    "show-fields":  "headline,trailText",
                    "api-key":      key
                }, timeout=10
            )
            if r.status_code == 200:
                return [{
                    "title":       a.get("fields", {}).get("headline", a.get("webTitle", "")),
                    "url":         a.get("webUrl", ""),
                    "publishedAt": a.get("webPublicationDate", ""),
                    "source":      {"name": "The Guardian"},
                    "description": a.get("fields", {}).get("trailText", ""),
                    "domain":      domain
                } for a in r.json().get("response", {}).get("results", [])], "guardian"
        except:
            pass
        return None, "error"

    # ════════════════════════════════════════
    # SMART BACKFILL FETCHER
    # ════════════════════════════════════════
    def backfill_with_fallback(topic, domain, from_date, to_date):
        # Try NewsAPI first
        results, status = backfill_newsapi(topic, domain, from_date, to_date)
        if results:
            return results, "newsapi"

        # NewsAPI failed — try GDELT + Guardian together
        gdelt_r,    _ = backfill_gdelt(topic, domain, from_date, to_date)
        guardian_r, _ = backfill_guardian(topic, domain, from_date, to_date)

        combined = []
        if gdelt_r:    combined.extend(gdelt_r)
        if guardian_r: combined.extend(guardian_r)
        if combined:
            return combined, "gdelt+guardian"

        # Last resort — GDELT alone
        gdelt_r, _ = backfill_gdelt(topic, domain, from_date, to_date)
        if gdelt_r:
            return gdelt_r, "gdelt"

        return [], "no_results"

    def scrape_fast(url):
        try:
            downloaded = trafilatura.fetch_url(url, config=traf_config)
            if downloaded:
                return trafilatura.extract(downloaded, config=traf_config)
        except:
            pass
        return None

    # ════════════════════════════════════════
    # RUN BACKFILL WEEK BY WEEK
    # ════════════════════════════════════════
    for from_date, to_date, week_key in to_fetch:
        print(f"📥 Fetching {week_key} ({from_date} → {to_date})...")

        week_articles = []
        seen_urls     = set()
        source_log    = defaultdict(int)

        for topic, domain in BACKFILL_TOPICS:
            results, source = backfill_with_fallback(topic, domain, from_date, to_date)
            for a in results:
                url = a.get("url", "")
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    week_articles.append(a)
                    source_log[source] += 1
            time.sleep(0.15)

        if not week_articles:
            print(f"   ⚠️  No articles from any source — skipping")
            continue

        # Source breakdown
        for src, cnt in source_log.items():
            print(f"   📰 {src}: {cnt} articles")
        print(f"   📰 Total: {len(week_articles)} articles — scraping full text...")

        # Parallel scrape
        urls   = [a.get("url", "") for a in week_articles if a.get("url")]
        ftexts = {}
        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            fmap = {executor.submit(scrape_fast, u): u for u in urls}
            for future in concurrent.futures.as_completed(fmap):
                u = fmap[future]
                try:
                    t = future.result(timeout=8)
                    if t: ftexts[u] = t
                except:
                    pass
        print(f"   🌐 {len(ftexts)}/{len(urls)} full text scraped")

        # Extract entities
        week_entities = []
        for article in week_articles:
            url  = article.get("url", "")
            text = ftexts.get(url) or \
                   f"{article.get('title', '')} {article.get('description', '')}"
            doc  = nlp(text[:5000])
            ents = []
            seen = set()
            for ent in doc.ents:
                if ent.label_ not in VALID_LABELS: continue
                n = normalize_entity(ent.text.strip())
                if len(n) < 2 or len(n) > 40 or n in seen: continue
                seen.add(n)
                ents.append((n, ent.label_))
            week_entities.append(ents)

        # Collect pairs
        pair_counts  = defaultdict(int)
        pair_domains = defaultdict(lambda: defaultdict(int))
        for article, ents in zip(week_articles, week_entities):
            unique = list(set([e[0] for e in ents]))
            domain = article.get("domain", "GENERAL")
            for i in range(len(unique)):
                for j in range(i + 1, len(unique)):
                    key = tuple(sorted([unique[i], unique[j]]))
                    pair_counts[key]          += 1
                    pair_domains[key][domain] += 1

        # Write to Neo4j
        with driver.session() as session:
            # Articles
            session.run("""
                UNWIND $batch AS row
                MERGE (a:Article {url: row.url})
                SET a.title=row.title, a.source=row.source,
                    a.published=row.published, a.week=row.week,
                    a.domain=row.domain
            """, {"batch": [{
                "url":       a.get("url", "unknown"),
                "title":     a.get("title", ""),
                "source":    a.get("source", {}).get("name", "Unknown"),
                "published": a.get("publishedAt", ""),
                "week":      week_key,
                "domain":    a.get("domain", "GENERAL")
            } for a in week_articles]})

            # Mentions
            mentions = []
            for article, ents in zip(week_articles, week_entities):
                for name, label in ents:
                    mentions.append({
                        "url":  article.get("url", "unknown"),
                        "name": name,
                        "type": label
                    })
            for i in range(0, len(mentions), 500):
                session.run("""
                    UNWIND $batch AS row
                    MERGE (e:Entity {name: row.name}) SET e.type = row.type
                    WITH e, row
                    MATCH (a:Article {url: row.url})
                    MERGE (a)-[:MENTIONS]->(e)
                """, {"batch": mentions[i:i+500]})

            # Pairs
            pairs_list = list(pair_counts.items())
            for bs in range(0, len(pairs_list), 200):
                batch = pairs_list[bs:bs + 200]
                session.run("""
                    UNWIND $batch AS row
                    MATCH (a:Entity {name: row.e1})
                    MATCH (b:Entity {name: row.e2})
                    MERGE (a)-[r:CO_OCCURS_WITH]-(b)
                    ON CREATE SET
                        r.count              = row.cnt,
                        r.first_seen         = row.week,
                        r.weekly_counts_json = row.wcj,
                        r.domains_json       = row.dj
                    ON MATCH SET
                        r.count              = r.count + row.cnt,
                        r.weekly_counts_json = CASE
                            WHEN r.weekly_counts_json IS NULL
                            THEN row.wcj
                            ELSE r.weekly_counts_json + row.wcj_append
                        END,
                        r.domains_json       = row.dj
                """, {"batch": [{
                    "e1":         p[0],
                    "e2":         p[1],
                    "cnt":        c,
                    "week":       week_key,
                    "wcj":        json.dumps({week_key: c}),
                    "wcj_append": f',"{week_key}":{c}',
                    "dj":         json.dumps(dict(pair_domains[p]))
                } for p, c in batch]})

        print(f"   ✅ {week_key} done — {len(week_articles)} articles, {len(pair_counts)} pairs\n")
        time.sleep(2)

    print("🎉 Backfill complete!\n")

# ── Final summary ──
print("📅 WEEKS NOW IN GRAPH:")
with driver.session() as session:
    for r in session.run("""
        MATCH (a:Article)
        RETURN a.week as week, count(a) as c
        ORDER BY week
    """):
        print(f"   {r['week']}: {r['c']} articles")

📅 Weeks to backfill: ['2026-W05', '2026-W06', '2026-W07', '2026-W08']
⏭️  Already have:     {'2026-W04', '2026-W08', '2026-W07', '2026-W05', '2026-W09', '2026-W06'}
📥 Will fetch:        []

✅ All weeks already in graph — nothing to backfill
📅 WEEKS NOW IN GRAPH:
   2026-W04: 158 articles
   2026-W05: 163 articles
   2026-W06: 174 articles
   2026-W07: 143 articles
   2026-W08: 101 articles
   2026-W09: 108 articles


In [ ]:
# ============================================
# PRAJNA — Cell 6: Test Groq + Graph Explorer
# ============================================

# ── Test Groq ──
def ask_groq(prompt):
    r = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":prompt}],
        max_tokens=600, temperature=0.3
    )
    return r.choices[0].message.content

print("🧪 Testing Groq connection...")
test = ask_groq("In one sentence, what is India's most important strategic relationship?")
print(f"✅ Groq: {test}\n")

# ── Graph explorer ──
with driver.session() as session:
    print("🌍 TOP 15 ENTITIES:")
    print("="*50)
    result = session.run("""
        MATCH (e:Entity)
        WITH e, COUNT{(e)--()} as conn
        ORDER BY conn DESC LIMIT 15
        RETURN e.name as name, e.type as type, conn
    """)
    for r in result:
        print(f"   {r['name']:<28} {r['type']:<10} {r['conn']}")

    print("\n📅 WEEKS IN GRAPH:")
    for r in session.run("""
        MATCH (a:Article) RETURN a.week as week, count(a) as c ORDER BY week
    """):
        print(f"   {r['week']}: {r['c']} articles")

    print("\n📈 SAMPLE TRAJECTORY — India ↔ Iran:")
    result = session.run("""
        MATCH (a:Entity {name:'India'})-[r:CO_OCCURS_WITH]-(b:Entity {name:'Iran'})
        RETURN r.weekly_counts_json AS wcj, r.count AS total
    """).single()
    if result and result["wcj"]:
        wc = json.loads(result["wcj"])
        for week, count in sorted(wc.items()):
            bar = "█" * min(count, 40)
            print(f"   {week}: {bar} ({count})")
    else:
        print("   No trajectory data yet — run backfill first")

🧪 Testing Groq connection...
✅ Groq: India's most important strategic relationship is with the United States, as the two countries have been strengthening their partnership in recent years through various initiatives, including defense cooperation, trade, and counter-terrorism efforts, to counterbalance China's growing influence in the region.

🌍 TOP 15 ENTITIES:
   United States                GPE        5659
   United States                GPE        5606
   United Kingdom               GPE        3694
   Trump                        PERSON     3476
   Guardian                     ORG        2607
   Iran                         GPE        2257
   American                     NORP       1981
   London                       GPE        1974
   Israel                       NORP       1944
   Europe                       LOC        1812
   China                        GPE        1779
   Russia                       GPE        1731
   Australia                    GPE        1682
   England

In [ ]:
import re
with driver.session() as session:
    rows = session.run("""
        MATCH ()-[r:CO_OCCURS_WITH]-()
        WHERE r.weekly_counts_json IS NOT NULL
        RETURN id(r) as rid, r.weekly_counts_json as wcj
    """).data()

    fixes = []
    for row in rows:
        try:
            json.loads(row["wcj"])  # test if valid
        except:
            # corrupted — rebuild using regex
            pairs = re.findall(r'"(2026-W\d+)"\s*:\s*(\d+)', row["wcj"])
            if pairs:
                fixes.append({
                    "rid": row["rid"],
                    "wcj": json.dumps({k: int(v) for k, v in pairs})
                })

    if fixes:
        for i in range(0, len(fixes), 500):
            session.run("""
                UNWIND $batch AS row
                MATCH ()-[r]-() WHERE id(r) = row.rid
                SET r.weekly_counts_json = row.wcj
            """, {"batch": fixes[i:i+500]})
        print(f"✅ Fixed {len(fixes)} corrupted relationships")
    else:
        print("✅ No corrupted JSON found")

✅ Fixed 19466 corrupted relationships


In [ ]:
# ============================================
# PRAJNA — Cell 7: PyVis Graph Visualization
# Run anytime to generate interactive graph
# ============================================

from pyvis.network import Network

def build_graph_html(keyword=None, limit=40):
    TYPE_COLORS = {
        "GPE":    "#C8A96E",
        "ORG":    "#6B8CAE",
        "PERSON": "#A8C5A0",
        "NORP":   "#B8956A",
        "LOC":    "#8AABBA",
        "EVENT":  "#C47B6E"
    }
    with driver.session() as session:
        if keyword:
            res = session.run("""
                MATCH (e:Entity) WHERE toLower(e.name) CONTAINS $kw
                WITH e MATCH (e)-[r:CO_OCCURS_WITH]-(other:Entity)
                WITH collect(DISTINCT e)+collect(DISTINCT other) AS nl
                UNWIND nl AS node WITH DISTINCT node
                WITH node, COUNT{(node)--()} AS conn
                ORDER BY conn DESC LIMIT $limit
                RETURN node.name AS name, node.type AS type, conn
            """, {"kw": keyword.lower(), "limit": limit})
        else:
            res = session.run("""
                MATCH (e:Entity)
                WITH e, COUNT{(e)--()} AS conn
                ORDER BY conn DESC LIMIT $limit
                RETURN e.name AS name, e.type AS type, conn
            """, {"limit": limit})
        top  = {r["name"]: r for r in res}
        rels = session.run("""
            MATCH (e1:Entity)-[r:CO_OCCURS_WITH]-(e2:Entity)
            WHERE e1.name IN $names AND e2.name IN $names AND r.count >= 2
            RETURN e1.name AS source, e2.name AS target, r.count AS weight
        """, {"names": list(top.keys())})
        relationships = list(rels)

    net = Network(height="600px", width="100%",
                  bgcolor="#0A0C0F", font_color="#8A9BB0",
                  notebook=True, cdn_resources="in_line")
    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -60,
          "centralGravity": 0.008,
          "springLength": 140,
          "springConstant": 0.06
        },
        "solver": "forceAtlas2Based",
        "stabilization": {"iterations": 120}
      },
      "edges": {
        "smooth": {"type": "continuous"},
        "color": {"opacity": 0.4}
      }
    }
    """)
    for name, data in top.items():
        is_india = (name == "India")
        color    = "#E8D5A3" if is_india else TYPE_COLORS.get(data["type"], "#5A6A7A")
        size     = 58 if is_india else min(12 + data["conn"] * 1.8, 52)
        net.add_node(
            name, label=name,
            color={"background": color, "border": "#FFFFFF" if is_india else color,
                   "highlight": {"background": "#E8D5A3", "border": "#FFFFFF"}},
            size=size,
            title=f"{data['type']} · {data['conn']} connections",
            font={"size": 11, "color": "#C8D4E0", "face": "IBM Plex Mono"}
        )
    for r in relationships:
        if r["source"] in top and r["target"] in top:
            net.add_edge(
                r["source"], r["target"],
                value=min(r["weight"] * 0.4, 4),
                title=f"co-occurrence: {r['weight']}",
                color={"color": "#2A3A4A", "highlight": "#C8A96E"}
            )

    net.save_graph("prajna_graph.html")
    print(f"✅ Graph saved — {len(top)} nodes, {len(relationships)} edges")
    return "prajna_graph.html"

# Generate and display
build_graph_html()

from IPython.display import IFrame
IFrame("prajna_graph.html", width="100%", height="620px")

✅ Graph saved — 39 nodes, 1408 edges


In [ ]:
# ============================================
# PRAJNA — Cell 8: Write Streamlit App
# Complete clean version with week selector
# ============================================

import base64

# ── Embed logo ──
try:
    with open('prajna_logo_v2.svg', 'rb') as f:
        LOGO_B64 = base64.b64encode(f.read()).decode()
    print("✅ Logo loaded")
except:
    LOGO_B64 = ""
    print("⚠️  Logo not found — upload prajna_logo_v2.svg to Colab first")

app_code = f'''
import streamlit as st
from neo4j import GraphDatabase
from groq import Groq
from pyvis.network import Network
import streamlit.components.v1 as components
import os, json, pandas as pd
from collections import defaultdict
from datetime import datetime, timezone

# ── Connections ──
NEO4J_URI      = os.environ.get("NEO4J_URI")      or st.secrets.get("NEO4J_URI")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME") or st.secrets.get("NEO4J_USERNAME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD") or st.secrets.get("NEO4J_PASSWORD")
GROQ_API_KEY   = os.environ.get("GROQ_API_KEY")   or st.secrets.get("GROQ_API_KEY")

LOGO_B64 = "{LOGO_B64}"

@st.cache_resource
def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

@st.cache_resource
def get_groq():
    return Groq(api_key=GROQ_API_KEY)

driver      = get_driver()
groq_client = get_groq()

def get_week_key():
    now = datetime.now(timezone.utc)
    return f"{{now.year}}-W{{now.strftime('%W')}}"

def ask_groq(prompt):
    r = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{{"role": "user", "content": prompt}}],
        max_tokens=600, temperature=0.3
    )
    return r.choices[0].message.content

@st.cache_data(ttl=300)
def get_stats():
    with driver.session() as session:
        a = session.run("MATCH (a:Article) RETURN count(a) as c").single()["c"]
        e = session.run("MATCH (e:Entity)  RETURN count(e) as c").single()["c"]
        r = session.run("MATCH ()-[r:CO_OCCURS_WITH]->() RETURN count(r) as c").single()["c"]
    return a, e, r

@st.cache_data(ttl=300)
def get_available_weeks():
    with driver.session() as session:
        result = session.run("""
            MATCH (a:Article)
            RETURN DISTINCT a.week as week
            ORDER BY week DESC
        """).data()
    return [r["week"] for r in result if r["week"]]

def get_graph_context(keywords):
    with driver.session() as session:
        result = session.run("""
            MATCH (e:Entity)-[r:CO_OCCURS_WITH]-(other:Entity)
            WHERE any(kw IN $kw WHERE toLower(e.name) CONTAINS toLower(kw))
            RETURN e.name AS e1, other.name AS e2, r.count AS strength
            ORDER BY r.count DESC LIMIT 20
        """, {{"kw": keywords}})
        return [(r["e1"], r["e2"], r["strength"]) for r in result]

def get_articles(keywords):
    with driver.session() as session:
        result = session.run("""
            MATCH (a:Article)-[:MENTIONS]->(e:Entity)
            WHERE any(kw IN $kw WHERE toLower(e.name) CONTAINS toLower(kw))
            RETURN DISTINCT a.title AS title, a.source AS source LIMIT 5
        """, {{"kw": keywords}})
        return [(r["title"], r["source"]) for r in result]

def get_trajectory(e1, e2):
    with driver.session() as session:
        result = session.run("""
            MATCH (a:Entity {{name:$e1}})-[r:CO_OCCURS_WITH]-(b:Entity {{name:$e2}})
            RETURN r.weekly_counts_json AS wcj, r.count AS total
        """, {{"e1": e1, "e2": e2}}).single()
    if not result or not result["wcj"]:
        return None, None
    wc = json.loads(result["wcj"])
    return sorted(wc.items()), result["total"]

def find_path(e1, e2):
    with driver.session() as session:
        result = session.run("""
            MATCH path = shortestPath(
                (a:Entity {{name:$e1}})-[:CO_OCCURS_WITH*1..4]-(b:Entity {{name:$e2}})
            )
            RETURN [node IN nodes(path) | node.name] AS nodes,
                   [rel IN relationships(path) | rel.count] AS strengths,
                   length(path) AS hops
        """, {{"e1": e1, "e2": e2}}).single()
    if not result:
        return None
    return {{"nodes": result["nodes"], "strengths": result["strengths"], "hops": result["hops"]}}

BLOCKLIST = {{
    "Supreme","Asian","Islam","American","European","Western","Eastern",
    "BBC","CNN","Reuters","Bloomberg","NDTV","Mint","Hindu","Express",
    "ANI","PTI","AFP","AP","Wire","Tribune","Times","Globe","Newswire"
}}

def detect_surges(threshold=1.5, top_n=5):
    week_key = get_week_key()
    with driver.session() as session:
        results = session.run("""
            MATCH (e:Entity)-[r:CO_OCCURS_WITH]-()
            WHERE r.weekly_counts_json IS NOT NULL
            RETURN e.name AS entity, e.type AS type,
                   r.weekly_counts_json AS wcj, r.count AS total
        """).data()

    entity_weekly = defaultdict(lambda: defaultdict(int))
    entity_total  = defaultdict(int)
    entity_type   = {{}}
    for row in results:
        e = row["entity"]
        entity_type[e]  = row["type"]
        entity_total[e] += row["total"] or 0
        wc = json.loads(row["wcj"]) if row["wcj"] else {{}}
        for week, count in wc.items():
            entity_weekly[e][week] += count

    surges = []
    for entity, weekly in entity_weekly.items():
        if entity in BLOCKLIST:                                      continue
        if entity_total[entity] < 8:                                 continue
        if len(entity) > 35:                                         continue
        if any(c.isdigit() for c in entity):                         continue
        if entity_type.get(entity) not in {{"GPE","ORG","PERSON","NORP"}}: continue
        weeks = sorted(weekly.keys())
        if not weeks or weeks[-1] != week_key:                       continue
        latest   = weekly[weeks[-1]]
        hist     = [weekly[w] for w in weeks[:-1]]
        baseline = sum(hist)/len(hist) if hist else entity_total[entity]/2
        if baseline == 0: continue
        ratio = latest / baseline
        if ratio >= threshold:
            surges.append({{
                "entity":   entity,
                "type":     entity_type.get(entity, ""),
                "latest":   latest,
                "baseline": round(baseline, 1),
                "ratio":    round(ratio, 2),
                "total":    entity_total[entity]
            }})
    surges.sort(key=lambda x: x["ratio"], reverse=True)
    return surges[:top_n]

def build_graph_visual(keyword=None, week=None):
    TYPE_COLORS = {{
        "GPE":    "#C8A96E",
        "ORG":    "#6B8CAE",
        "PERSON": "#A8C5A0",
        "NORP":   "#B8956A",
        "LOC":    "#8AABBA",
        "EVENT":  "#C47B6E"
    }}
    with driver.session() as session:
        # ── Get top nodes ──
        if week:
            if keyword:
                res = session.run("""
                    MATCH (a:Article {{week:$week}})-[:MENTIONS]->(e:Entity)
                    WHERE toLower(e.name) CONTAINS $kw
                    WITH e MATCH (e)-[:CO_OCCURS_WITH]-(other:Entity)
                    WITH collect(DISTINCT e)+collect(DISTINCT other) AS nl
                    UNWIND nl AS node WITH DISTINCT node
                    WITH node, COUNT{{(node)--()}} AS conn
                    ORDER BY conn DESC LIMIT 40
                    RETURN node.name AS name, node.type AS type, conn
                """, {{"week": week, "kw": keyword.lower()}})
            else:
                res = session.run("""
                    MATCH (a:Article {{week:$week}})-[:MENTIONS]->(e:Entity)
                    WITH e, COUNT{{(e)--()}} AS conn
                    ORDER BY conn DESC LIMIT 40
                    RETURN e.name AS name, e.type AS type, conn
                """, {{"week": week}})
        else:
            if keyword:
                res = session.run("""
                    MATCH (e:Entity) WHERE toLower(e.name) CONTAINS $kw
                    WITH e MATCH (e)-[:CO_OCCURS_WITH]-(other:Entity)
                    WITH collect(DISTINCT e)+collect(DISTINCT other) AS nl
                    UNWIND nl AS node WITH DISTINCT node
                    WITH node, COUNT{{(node)--()}} AS conn
                    ORDER BY conn DESC LIMIT 40
                    RETURN node.name AS name, node.type AS type, conn
                """, {{"kw": keyword.lower()}})
            else:
                res = session.run("""
                    MATCH (e:Entity)
                    WITH e, COUNT{{(e)--()}} AS conn
                    ORDER BY conn DESC LIMIT 40
                    RETURN e.name AS name, e.type AS type, conn
                """)
        top = {{r["name"]: r for r in res}}

        # ── Get edges — use weekly counts if week selected ──
        if week:
            rels = session.run("""
                MATCH (e1:Entity)-[r:CO_OCCURS_WITH]-(e2:Entity)
                WHERE e1.name IN $names AND e2.name IN $names
                AND r.weekly_counts_json IS NOT NULL
                RETURN e1.name AS source, e2.name AS target,
                       r.weekly_counts_json AS wcj
            """, {{"names": list(top.keys())}})
            relationships = []
            for r in rels:
                wc = json.loads(r["wcj"]) if r["wcj"] else {{}}
                w  = wc.get(week, 0)
                if w >= 1:
                    relationships.append({{
                        "source": r["source"],
                        "target": r["target"],
                        "weight": w
                    }})
        else:
            rels = session.run("""
                MATCH (e1:Entity)-[r:CO_OCCURS_WITH]-(e2:Entity)
                WHERE e1.name IN $names AND e2.name IN $names AND r.count >= 2
                RETURN e1.name AS source, e2.name AS target, r.count AS weight
            """, {{"names": list(top.keys())}})
            relationships = [dict(r) for r in rels]

    # ── Build PyVis ──
    net = Network(height="460px", width="100%", bgcolor="#0A0C0F",
                  font_color="#8A9BB0", notebook=False, cdn_resources="in_line")
    net.set_options("""
    {{
      "physics": {{
        "forceAtlas2Based": {{
          "gravitationalConstant": -60,
          "centralGravity": 0.008,
          "springLength": 140,
          "springConstant": 0.06
        }},
        "solver": "forceAtlas2Based",
        "stabilization": {{"iterations": 120}}
      }},
      "edges": {{
        "smooth": {{"type": "continuous"}},
        "color": {{"opacity": 0.4}}
      }}
    }}
    """)

    for name, data in top.items():
        is_india = (name == "India")
        color    = "#E8D5A3" if is_india else TYPE_COLORS.get(data["type"], "#5A6A7A")
        size     = 58 if is_india else min(12 + data["conn"] * 1.8, 52)
        net.add_node(name, label=name,
            color={{
                "background":  color,
                "border":      "#FFFFFF" if is_india else color,
                "highlight":   {{"background": "#E8D5A3", "border": "#FFFFFF"}}
            }},
            size=size,
            title=f"{{data['type']}} · {{data['conn']}} connections",
            font={{"size": 11, "color": "#C8D4E0", "face": "IBM Plex Mono"}})

    for r in relationships:
        if r["source"] in top and r["target"] in top:
            net.add_edge(r["source"], r["target"],
                value=min(r["weight"] * 0.4, 4),
                title=f"co-occurrence: {{r['weight']}}",
                color={{"color": "#2A3A4A", "highlight": "#C8A96E"}})

    net.save_graph("graph_visual.html")
    with open("graph_visual.html", "r") as f:
        return f.read()


# ════════════════════════════════════════════
# PAGE CONFIG
# ════════════════════════════════════════════
st.set_page_config(
    page_title="PRAJNA — Strategic Intelligence",
    page_icon="▪",
    layout="wide",
    initial_sidebar_state="collapsed"
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@300;400;500;600&family=IBM+Plex+Sans:wght@300;400;500;600&display=swap');

*, *::before, *::after {{ box-sizing: border-box; }}
html, body, .stApp {{
    background-color: #0A0C0F !important;
    color: #C8D4E0 !important;
    font-family: 'IBM Plex Sans', sans-serif !important;
}}
#MainMenu, footer, header {{ visibility: hidden; }}
.stDeployButton {{ display: none; }}
section[data-testid="stSidebar"] {{ display: none; }}

.prajna-nav {{
    display: flex; align-items: center;
    justify-content: space-between;
    padding: 0 0 20px 0;
    border-bottom: 1px solid #1C2A38;
    margin-bottom: 20px;
}}
.prajna-logo {{
    font-family: 'IBM Plex Mono', monospace;
    font-size: 13px; font-weight: 600;
    letter-spacing: 0.25em; color: #E8D5A3; text-transform: uppercase;
}}
.prajna-logo span {{ color: #4A6A8A; margin: 0 12px; }}
.prajna-meta {{
    font-family: 'IBM Plex Mono', monospace;
    font-size: 10px; color: #3A5068;
    letter-spacing: 0.1em; text-transform: uppercase;
}}
.prajna-status {{
    display: flex; align-items: center; gap: 6px;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 10px; color: #4A8A6A; letter-spacing: 0.1em;
}}
.status-dot {{
    width: 6px; height: 6px; background: #4A8A6A;
    border-radius: 50%; animation: pulse 2s infinite;
}}
@keyframes pulse {{ 0%,100% {{ opacity:1; }} 50% {{ opacity:0.3; }} }}

.stTabs [data-baseweb="tab-list"] {{
    background: transparent !important;
    border-bottom: 1px solid #1C2A38 !important;
    gap: 0 !important; padding: 0 !important;
}}
.stTabs [data-baseweb="tab"] {{
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 10px !important; font-weight: 500 !important;
    letter-spacing: 0.15em !important; text-transform: uppercase !important;
    color: #3A5068 !important; background: transparent !important;
    border: none !important; border-bottom: 2px solid transparent !important;
    padding: 10px 20px !important; border-radius: 0 !important;
}}
.stTabs [aria-selected="true"] {{
    color: #E8D5A3 !important;
    border-bottom: 2px solid #E8D5A3 !important;
    background: transparent !important;
}}
.stTabs [data-baseweb="tab-panel"] {{
    padding-top: 24px !important; background: transparent !important;
}}

.stTextInput > div > div > input {{
    background: #0D1117 !important; border: 1px solid #1C2A38 !important;
    border-radius: 0 !important; color: #C8D4E0 !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 13px !important; padding: 10px 14px !important;
}}
.stTextInput > div > div > input:focus {{
    border-color: #C8A96E !important; box-shadow: none !important;
}}
.stTextInput label {{
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 9px !important; text-transform: uppercase !important;
    letter-spacing: 0.15em !important; color: #3A5068 !important;
}}

.stSelectbox > div > div {{
    background: #0D1117 !important; border: 1px solid #1C2A38 !important;
    border-radius: 0 !important; color: #C8D4E0 !important;
    font-family: 'IBM Plex Mono', monospace !important; font-size: 11px !important;
}}
.stSelectbox label {{
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 9px !important; text-transform: uppercase !important;
    letter-spacing: 0.15em !important; color: #3A5068 !important;
}}

.stButton > button {{
    background: transparent !important; border: 1px solid #2A3A4A !important;
    border-radius: 0 !important; color: #8A9BB0 !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 10px !important; font-weight: 500 !important;
    letter-spacing: 0.15em !important; text-transform: uppercase !important;
    padding: 8px 20px !important; transition: all 0.15s ease !important;
}}
.stButton > button:hover {{
    background: #1C2A38 !important; border-color: #C8A96E !important;
    color: #E8D5A3 !important;
}}
.stButton > button:focus {{ box-shadow: none !important; outline: none !important; }}

.stSlider > div > div > div {{ background: #1C2A38 !important; }}
.stSlider label {{
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 9px !important; text-transform: uppercase !important;
    letter-spacing: 0.15em !important; color: #3A5068 !important;
}}

.brief-container {{
    border: 1px solid #1C2A38; border-left: 3px solid #C8A96E;
    background: #0D1117; padding: 20px 24px; margin: 16px 0;
    font-family: 'IBM Plex Sans', sans-serif; font-size: 13px;
    line-height: 1.8; color: #A8B8C8;
}}
.brief-header {{
    font-family: 'IBM Plex Mono', monospace; font-size: 9px;
    color: #C8A96E; text-transform: uppercase; letter-spacing: 0.2em;
    margin-bottom: 14px; padding-bottom: 10px; border-bottom: 1px solid #1C2A38;
}}
.path-node {{
    display: flex; align-items: center; gap: 12px;
    padding: 10px 16px; margin: 2px 0;
    background: #0D1117; border: 1px solid #1C2A38;
    font-family: 'IBM Plex Mono', monospace; font-size: 12px; color: #C8D4E0;
}}
.path-node.start {{ border-left: 3px solid #6B8CAE; }}
.path-node.end   {{ border-left: 3px solid #C8A96E; }}
.path-node.mid   {{ border-left: 3px solid #2A3A4A; margin-left: 20px; }}
.path-connector  {{
    margin-left: 28px; padding: 4px 16px;
    font-family: 'IBM Plex Mono', monospace; font-size: 10px;
    color: #2A4A3A; letter-spacing: 0.1em;
}}
.source-tag {{
    font-family: 'IBM Plex Mono', monospace; font-size: 9px;
    color: #3A5068; border: 1px solid #1C2A38;
    padding: 3px 8px; letter-spacing: 0.08em; text-transform: uppercase;
}}
.surge-card {{
    border: 1px solid #1C2A38; border-left: 3px solid #8A4A3A;
    background: #0D1117; padding: 16px 20px; margin: 8px 0;
}}
.surge-entity {{
    font-family: 'IBM Plex Mono', monospace; font-size: 13px;
    font-weight: 600; color: #E8D5A3;
    letter-spacing: 0.1em; text-transform: uppercase;
}}
.surge-ratio {{
    font-family: 'IBM Plex Mono', monospace; font-size: 11px;
    color: #C87B6A; margin-top: 2px;
}}
.section-label {{
    font-family: 'IBM Plex Mono', monospace; font-size: 9px;
    color: #3A5068; text-transform: uppercase; letter-spacing: 0.2em;
    margin-bottom: 12px; padding-bottom: 8px; border-bottom: 1px solid #1C2A38;
}}
div[data-testid="metric-container"] {{
    background: #0D1117 !important; border: 1px solid #1C2A38 !important;
    border-radius: 0 !important; padding: 12px 16px !important;
}}
div[data-testid="metric-container"] label {{
    font-family: 'IBM Plex Mono', monospace !important; font-size: 9px !important;
    text-transform: uppercase !important; letter-spacing: 0.15em !important;
    color: #3A5068 !important;
}}
div[data-testid="metric-container"] div[data-testid="stMetricValue"] {{
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 20px !important; color: #E8D5A3 !important;
}}
::-webkit-scrollbar {{ width: 4px; }}
::-webkit-scrollbar-track {{ background: #0A0C0F; }}
::-webkit-scrollbar-thumb {{ background: #1C2A38; }}
.stSpinner > div {{ border-top-color: #C8A96E !important; }}
.stAlert {{
    background: #0D1117 !important; border: 1px solid #1C2A38 !important;
    border-radius: 0 !important; font-family: 'IBM Plex Mono', monospace !important;
    font-size: 11px !important; color: #5A7A9A !important;
}}
</style>
""", unsafe_allow_html=True)


# ════════════════════════════════════════════
# NAV BAR
# ════════════════════════════════════════════
now_str = datetime.now(timezone.utc).strftime("%Y-%m-%d  %H:%M UTC")

try:
    articles_n, entities_n, rels_n = get_stats()
except:
    articles_n, entities_n, rels_n = 0, 0, 0

logo_html = f\'<img src="data:image/svg+xml;base64,{{LOGO_B64}}" width="52" height="52" style="opacity:0.95"/>\' if LOGO_B64 else ""

st.markdown(f"""
<div class="prajna-nav">
    <div style="display:flex;align-items:center;gap:16px">
        {{logo_html}}
        <div>
            <div class="prajna-logo">PRAJNA<span>▪</span>Strategic Intelligence Engine</div>
            <div class="prajna-meta" style="margin-top:4px">
                India Innovates 2026 &nbsp;·&nbsp; Domain 02: Digital Democracy &nbsp;·&nbsp; TeamIIMC
            </div>
        </div>
    </div>
    <div style="text-align:right">
        <div class="prajna-status">
            <div class="status-dot"></div>GRAPH ACTIVE
        </div>
        <div class="prajna-meta" style="margin-top:4px">{{now_str}}</div>
    </div>
</div>
""", unsafe_allow_html=True)

# Stats row
c1, c2, c3, c4 = st.columns(4)
c1.metric("ARTICLES INGESTED", articles_n)
c2.metric("ENTITIES MAPPED",   entities_n)
c3.metric("RELATIONSHIPS",     rels_n)
c4.metric("GRAPH STATUS",      "● LIVE")

st.markdown("<div style='margin-bottom:8px'></div>", unsafe_allow_html=True)


# ════════════════════════════════════════════
# TABS
# ════════════════════════════════════════════
tab1, tab2, tab3, tab4 = st.tabs([
    "INTELLIGENCE BRIEF",
    "TRAJECTORY ANALYSIS",
    "PATH QUERY",
    "SURGE DETECTION"
])


# ── TAB 1: INTELLIGENCE BRIEF ──────────────
with tab1:
    col_left, col_right = st.columns([1, 1], gap="large")

    with col_left:
        st.markdown(\'<div class="section-label">Query Interface</div>\', unsafe_allow_html=True)

        if "query_input" not in st.session_state:
            st.session_state["query_input"] = ""

        query = st.text_input(
            "STRATEGIC QUESTION",
            placeholder="What are India\'s energy security risks from Iran?",
            value=st.session_state["query_input"]
        )

        st.markdown(\'<div style="margin:12px 0 8px;font-family:IBM Plex Mono;font-size:9px;color:#2A3A4A;letter-spacing:0.15em;text-transform:uppercase;">Suggested Queries</div>\', unsafe_allow_html=True)

        demo_queries = [
            "India Iran energy security",
            "Pakistan Afghanistan India impact",
            "India China border tensions",
            "India Russia defense cooperation",
            "Israel Iran war implications",
        ]
        for dq in demo_queries:
            if st.button(dq.upper(), key=f"dq_{{dq}}"):
                st.session_state["query_input"] = dq
                st.rerun()

        st.markdown("<div style='margin-top:16px'></div>", unsafe_allow_html=True)
        run_query = st.button("EXECUTE QUERY ▶", key="run_brief")

        if run_query and query:
            with st.spinner(""):
                keywords     = [w for w in query.lower().split() if len(w) > 3][:5]
                context      = get_graph_context(keywords)
                articles_ctx = get_articles(keywords)
                ctx_str      = "\\n".join([f"  {{e1}} <-> {{e2}} (strength:{{s}})" for e1, e2, s in context])
                art_str      = "\\n".join([f"  [{{src}}] {{title}}" for title, src in articles_ctx])

                prompt = f"""You are Prajna, India\'s strategic intelligence engine.
Answer ONLY from the graph context below. Cite every claim with its source.

KNOWLEDGE GRAPH CONNECTIONS:
{{ctx_str}}

NEWS SOURCES:
{{art_str}}

QUESTION: {{query}}

Structure your response as:
SITUATION — 2-3 sentence summary of the current state
KEY CONNECTIONS — 3-4 bullet points directly from the graph data
STRATEGIC IMPLICATIONS — what this means specifically for India
RECOMMENDED ACTION — one concrete action for Indian policymakers
SOURCES — list all cited sources"""

                brief = ask_groq(prompt)

            sources_html = ""
            if articles_ctx:
                tags = "".join([f\'<span class="source-tag">{{src}}</span>\' for _, src in articles_ctx[:6]])
                sources_html = f\'<div style="display:flex;flex-wrap:wrap;gap:6px;margin-top:12px;padding-top:12px;border-top:1px solid #1C2A38">{{tags}}</div>\'

            st.markdown(f"""
            <div class="brief-container">
                <div class="brief-header">
                    ▪ INTELLIGENCE BRIEF &nbsp;·&nbsp;
                    {{datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}}
                    &nbsp;·&nbsp; {{len(context)}} graph connections analysed
                </div>
                {{brief.replace(chr(10), "<br>")}}
                {{sources_html}}
            </div>
            """, unsafe_allow_html=True)

        elif run_query and not query:
            st.warning("Enter a strategic question first.")

    with col_right:
        st.markdown(\'<div class="section-label">Knowledge Graph</div>\', unsafe_allow_html=True)

        # ── Week selector + entity filter ──
        available_weeks = get_available_weeks()
        week_options    = ["ALL TIME"] + available_weeks

        c_filter, c_week = st.columns([2, 1])
        with c_filter:
            graph_filter = st.text_input(
                "FILTER BY ENTITY",
                placeholder="Iran, China, Taliban...",
                key="graph_filter"
            )
        with c_week:
            selected_week = st.selectbox(
                "WEEK VIEW",
                week_options,
                key="graph_week"
            )

        week_param = None if selected_week == "ALL TIME" else selected_week

        with st.spinner(""):
            try:
                graph_html = build_graph_visual(
                    keyword=graph_filter or None,
                    week=week_param
                )
                components.html(graph_html, height=460)
                week_label = f"Showing: {{selected_week}}" if week_param else "Showing: All time"
                st.markdown(f\'<div style="font-family:IBM Plex Mono;font-size:9px;color:#2A3A4A;text-transform:uppercase;letter-spacing:0.1em;margin-top:8px">{{week_label}} &nbsp;·&nbsp; Node size = connection strength &nbsp;·&nbsp; Drag to explore &nbsp;·&nbsp; Hover for details</div>\', unsafe_allow_html=True)
            except Exception as e:
                st.error(f"Graph error: {{e}}")


# ── TAB 2: TRAJECTORY ANALYSIS ─────────────
with tab2:
    st.markdown(\'<div class="section-label">Relationship Trajectory Analysis</div>\', unsafe_allow_html=True)
    st.markdown(\'<div style="font-family:IBM Plex Mono;font-size:10px;color:#3A5068;margin-bottom:20px;line-height:1.7">Tracks co-occurrence strength between two entities week-over-week.<br>Sustained increases = deepening relationship. Spikes = event-driven. Drops = decoupling.</div>\', unsafe_allow_html=True)

    c1, c2, c3 = st.columns([2, 2, 1])
    with c1: e1 = st.text_input("ENTITY A", value="India", key="traj_e1")
    with c2: e2 = st.text_input("ENTITY B", value="Iran",  key="traj_e2")
    with c3:
        st.markdown("<div style='margin-top:28px'></div>", unsafe_allow_html=True)
        run_traj = st.button("RUN ANALYSIS ▶", key="run_traj")

    st.markdown(\'<div style="font-family:IBM Plex Mono;font-size:9px;color:#2A3A4A;letter-spacing:0.1em;margin-bottom:16px">SUGGESTED — India/Iran &nbsp;·&nbsp; India/Russia &nbsp;·&nbsp; Iran/Israel &nbsp;·&nbsp; India/China &nbsp;·&nbsp; Pakistan/Afghanistan</div>\', unsafe_allow_html=True)

    if run_traj:
        with st.spinner(""):
            weekly_data, total = get_trajectory(e1, e2)

        if not weekly_data:
            st.markdown(f\'<div style="font-family:IBM Plex Mono;font-size:11px;color:#5A3A3A;border:1px solid #2A1A1A;padding:14px 18px;background:#0D0A0A">▪ NO TRAJECTORY DATA — {{e1.upper()}} ↔ {{e2.upper()}}<br><span style="color:#2A3A4A;font-size:9px">These entities may not co-occur in graph. Try: India/Iran · Iran/Israel · India/Russia</span></div>\', unsafe_allow_html=True)
        else:
            st.markdown(f\'<div style="font-family:IBM Plex Mono;font-size:10px;color:#4A6A4A;border:1px solid #1A2A1A;padding:10px 16px;background:#0A0D0A;margin-bottom:16px">▪ {{e1.upper()}} ↔ {{e2.upper()}} &nbsp;·&nbsp; {{total}} TOTAL CO-OCCURRENCES &nbsp;·&nbsp; {{len(weekly_data)}} WEEK(S) OF DATA</div>\', unsafe_allow_html=True)

            df = pd.DataFrame(weekly_data, columns=["Week", "Co-occurrences"])
            st.bar_chart(df.set_index("Week"), color="#C8A96E")

            traj_str = "\\n".join([f"  {{w}}: {{c}}" for w, c in weekly_data])
            prompt = f"""You are Prajna. Analyse this relationship trajectory:

{{e1}} ↔ {{e2}}
Weekly co-occurrence data:
{{traj_str}}
Total all-time co-occurrences: {{total}}

SITUATION — is this relationship intensifying, cooling, or volatile?
INFLECTION POINTS — any sharp changes and their likely geopolitical causes
INDIA STRATEGIC IMPLICATIONS — what does this trajectory mean for India specifically?
WATCH FOR — 2 specific developments to monitor in coming weeks

Be analytical and specific. Max 250 words."""

            with st.spinner(""):
                brief = ask_groq(prompt)

            st.markdown(f\'<div class="brief-container"><div class="brief-header">▪ TRAJECTORY INTERPRETATION — {{e1.upper()}} ↔ {{e2.upper()}}</div>{{brief.replace(chr(10), "<br>")}}</div>\', unsafe_allow_html=True)


# ── TAB 3: PATH QUERY ──────────────────────
with tab3:
    st.markdown(\'<div class="section-label">Graph Path Query</div>\', unsafe_allow_html=True)
    st.markdown(\'<div style="font-family:IBM Plex Mono;font-size:10px;color:#3A5068;margin-bottom:20px;line-height:1.7">Finds the shortest connection path between any two entities in the knowledge graph.<br>Surfaces non-obvious relationships invisible to human analysts — impossible for standard LLMs.</div>\', unsafe_allow_html=True)

    c1, c2, c3 = st.columns([2, 2, 1])
    with c1: p1 = st.text_input("ORIGIN ENTITY",  value="India",   key="path_p1")
    with c2: p2 = st.text_input("TARGET ENTITY",  value="Taliban", key="path_p2")
    with c3:
        st.markdown("<div style='margin-top:28px'></div>", unsafe_allow_html=True)
        run_path = st.button("FIND PATH ▶", key="run_path")

    st.markdown(\'<div style="font-family:IBM Plex Mono;font-size:9px;color:#2A3A4A;letter-spacing:0.1em;margin-bottom:16px">SUGGESTED — India→Taliban &nbsp;·&nbsp; Russia→Taliban &nbsp;·&nbsp; Israel→Pakistan &nbsp;·&nbsp; India→Qatar</div>\', unsafe_allow_html=True)

    if run_path:
        with st.spinner(""):
            path = find_path(p1, p2)

        if not path:
            st.markdown(f\'<div style="font-family:IBM Plex Mono;font-size:11px;color:#5A3A3A;border:1px solid #2A1A1A;padding:14px 18px;background:#0D0A0A">▪ NO PATH FOUND — {{p1.upper()}} → {{p2.upper()}}<br><span style="color:#2A3A4A;font-size:9px">No connection within 4 hops. Try different entity names.</span></div>\', unsafe_allow_html=True)
        else:
            nodes     = path["nodes"]
            strengths = path["strengths"]

            st.markdown(f\'<div style="font-family:IBM Plex Mono;font-size:10px;color:#4A6A4A;border:1px solid #1A2A1A;padding:10px 16px;background:#0A0D0A;margin-bottom:16px">▪ PATH FOUND &nbsp;·&nbsp; {{path["hops"]}} HOP(S) &nbsp;·&nbsp; {{p1.upper()}} → {{p2.upper()}}</div>\', unsafe_allow_html=True)

            nodes_html = ""
            path_str   = ""
            for i, node in enumerate(nodes):
                if i == 0:
                    nodes_html += f\'<div class="path-node start">▶ &nbsp; {{node.upper()}}</div>\'
                    path_str   += node
                elif i == len(nodes) - 1:
                    nodes_html += f\'<div class="path-node end">◼ &nbsp; {{node.upper()}}</div>\'
                    path_str   += node
                else:
                    nodes_html += f\'<div class="path-node mid">◦ &nbsp; {{node}}</div>\'
                    path_str   += node
                if i < len(strengths):
                    nodes_html += f\'<div class="path-connector">│ co-occurs {{strengths[i]}}× in news</div>\'
                    path_str   += f" --[{{strengths[i]}}x]--> "

            st.markdown(nodes_html, unsafe_allow_html=True)

            prompt = f"""You are Prajna. A knowledge graph path query revealed:

PATH: {{path_str}}
HOPS: {{path["hops"]}}

Each arrow represents two entities co-appearing in real news articles.
The number = frequency of co-appearance in the corpus.

HIDDEN CONNECTION — what does this path mean in plain English?
STRATEGIC SIGNIFICANCE — why does this specifically matter for India?
NON-OBVIOUS INSIGHT — what would an analyst miss without this graph?
RECOMMENDED ACTION — one concrete action for Indian policymakers

Be direct and specific. Max 220 words."""

            with st.spinner(""):
                brief = ask_groq(prompt)

            st.markdown(f\'<div class="brief-container" style="margin-top:16px"><div class="brief-header">▪ PATH INTELLIGENCE — {{p1.upper()}} → {{p2.upper()}}</div>{{brief.replace(chr(10), "<br>")}}</div>\', unsafe_allow_html=True)


# ── TAB 4: SURGE DETECTION ─────────────────
with tab4:
    st.markdown(\'<div class="section-label">Automated Surge Detection</div>\', unsafe_allow_html=True)
    st.markdown(\'<div style="font-family:IBM Plex Mono;font-size:10px;color:#3A5068;margin-bottom:20px;line-height:1.7">Monitors the knowledge graph for entities whose news co-occurrence is growing anomalously fast.<br>No query required — Prajna surfaces what you did not know to ask about.</div>\', unsafe_allow_html=True)

    c1, c2 = st.columns([2, 1])
    with c1:
        threshold = st.slider("SURGE THRESHOLD (× WEEKLY BASELINE)", 1.2, 3.0, 1.5, 0.1)
    with c2:
        st.markdown("<div style='margin-top:28px'></div>", unsafe_allow_html=True)
        run_surge = st.button("SCAN GRAPH ▶", key="run_surge")

    if run_surge:
        with st.spinner(""):
            surges = detect_surges(threshold=threshold)

        if not surges:
            st.markdown(f\'<div style="font-family:IBM Plex Mono;font-size:11px;color:#3A5068;border:1px solid #1C2A38;padding:14px 18px;background:#0D1117;margin-bottom:16px">▪ NO SURGES DETECTED ABOVE {{threshold}}× THRESHOLD<br><span style="font-size:9px;color:#2A3A4A">Surge detection strengthens as more weeks accumulate in the graph.</span></div>\', unsafe_allow_html=True)

            # Show top entities by connection as fallback
            st.markdown(\'<div class="section-label" style="margin-top:20px">Current Top Entities by Connection Strength</div>\', unsafe_allow_html=True)
            with driver.session() as session:
                top = session.run("""
                    MATCH (e:Entity)-[r:CO_OCCURS_WITH]-()
                    WHERE e.type IN [\'GPE\',\'ORG\',\'PERSON\']
                    RETURN e.name AS name, e.type AS type, count(r) AS conn
                    ORDER BY conn DESC LIMIT 12
                """).data()
            rows = "".join([
                f\'<div style="display:flex;justify-content:space-between;align-items:center;padding:6px 12px;border-bottom:1px solid #0D1117;font-family:IBM Plex Mono;font-size:10px;color:#5A7A9A"><span>{{r["name"].upper()}}</span><span style="color:#3A5068">{{r["conn"]}} connections</span></div>\'
                for r in top if len(r["name"]) < 35
            ])
            st.markdown(f\'<div style="border:1px solid #1C2A38;background:#0D1117">{{rows}}</div>\', unsafe_allow_html=True)

        else:
            st.markdown(f\'<div style="font-family:IBM Plex Mono;font-size:10px;color:#C87B6A;border:1px solid #2A1A1A;padding:10px 16px;background:#0D0A0A;margin-bottom:16px">▪ {{len(surges)}} ENTITIES FLAGGED ABOVE {{threshold}}× BASELINE</div>\', unsafe_allow_html=True)

            for surge in surges:
                st.markdown(f\'<div class="surge-card"><div class="surge-entity">▲ {{surge["entity"]}}</div><div class="surge-ratio">{{surge["ratio"]}}× baseline &nbsp;·&nbsp; {{surge["latest"]}} connections this week &nbsp;·&nbsp; baseline: {{surge["baseline"]}}/week</div></div>\', unsafe_allow_html=True)

                prompt = f"""Prajna surge alert:
Entity: {{surge["entity"]}} ({{surge["type"]}})
This week: {{surge["latest"]}} news co-occurrences
Historical baseline: {{surge["baseline"]}} per week
Surge ratio: {{surge["ratio"]}}×

SURGE EXPLANATION — why is this entity surging in the news?
INDIA RISK/OPPORTUNITY — what are the strategic implications for India?
URGENCY — Immediate / Days / Weeks
RECOMMENDED ACTION — one specific action

Max 130 words. Be direct and analytical."""

                with st.spinner(""):
                    brief = ask_groq(prompt)

                st.markdown(f\'<div class="brief-container" style="border-left-color:#8A4A3A;margin-bottom:20px"><div class="brief-header" style="color:#C87B6A">▪ SURGE BRIEF — {{surge["entity"].upper()}}</div>{{brief.replace(chr(10), "<br>")}}</div>\', unsafe_allow_html=True)
'''

with open("prajna_app.py", "w") as f:
    f.write(app_code)

print("✅ prajna_app.py written successfully")
print(f"   Size: {len(app_code):,} characters")
print("\n📋 What's new in this version:")
print("   ✅ Week selector on knowledge graph (ALL TIME + per week)")
print("   ✅ Graph edges use weekly counts when week is selected")
print("   ✅ Suggested query buttons fixed (no session state conflict)")
print("   ✅ Stats as clean metric cards (no raw HTML)")
print("   ✅ Logo embedded from prajna_logo_v2.svg")
print("   ✅ All 4 tabs working")
print("\n⚠️  Make sure prajna_logo_v2.svg is uploaded to Colab before running this cell")

✅ Logo loaded
✅ prajna_app.py written successfully
   Size: 49,287 characters

📋 What's new in this version:
   ✅ Week selector on knowledge graph (ALL TIME + per week)
   ✅ Graph edges use weekly counts when week is selected
   ✅ Suggested query buttons fixed (no session state conflict)
   ✅ Stats as clean metric cards (no raw HTML)
   ✅ Logo embedded from prajna_logo_v2.svg
   ✅ All 4 tabs working

⚠️  Make sure prajna_logo_v2.svg is uploaded to Colab before running this cell


In [ ]:
# ============================================
# PRAJNA — Cell 9: Download + Launch
# ============================================

from google.colab import files

# Download app for GitHub
files.download('prajna_app.py')
print("✅ prajna_app.py downloaded — upload to GitHub")

# ── Optional: Launch locally via ngrok ──
# Uncomment below to run locally in Colab

# from pyngrok import ngrok
# import subprocess, os
# from google.colab import userdata
# os.environ["NEO4J_URI"]      = userdata.get("NEO4J_URI")
# os.environ["NEO4J_USERNAME"] = userdata.get("NEO4J_USERNAME")
# os.environ["NEO4J_PASSWORD"] = userdata.get("NEO4J_PASSWORD")
# os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_API_KEY")
# NGROK_TOKEN = "YOUR_NGROK_TOKEN"
# ngrok.set_auth_token(NGROK_TOKEN)
# ngrok.kill()
# process = subprocess.Popen(["streamlit","run","prajna_app.py",
#     "--server.port=8501","--server.headless=true",
#     "--server.enableCORS=false","--server.enableXsrfProtection=false"])
# import time; time.sleep(5)
# tunnel = ngrok.connect(8501)
# print(f"🌐 LOCAL URL: {tunnel.public_url}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ prajna_app.py downloaded — upload to GitHub
